# 19–20 | 주형 담당: XGB 클래스 가중치 + 절단 변이 상대 위치 Ablation

**기존 코드/파일 수정 없음. train만 사용, 팀 고정 Dev(4,960)와 동일 Group 5-Fold로 비교.**

- 실험 A: α = 0.5(기준), 0.75, 1.0
- 실험 B: 기준 α=0.5에서 상대 위치 4/6/8/10구간을 추가
- 기준 모델: 11번 `XGB E+Hotspot×3+Position-bin+Exact-position` CPU 구성. 기존 11번의 LR/g3 예측은 별도 검증 없이 합치지 않습니다.
- 각 fold 학습 데이터에서만 class weight, 상대 위치 최대값, 위치 변환기 fit
- Dev 점수는 11번 **전체 OOF 0.5293과 직접 비교 금지**. 동일 Dev·Fold의 기준과만 비교
- 실행 전에 `src/`, `scripts/fixed_holdout_manifest.py` 및 팀 데이터 경로가 정상이어야 합니다.


In [1]:
from pathlib import Path
import sys, json, warnings
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score
from xgboost import XGBClassifier

start = Path.cwd().resolve()
ROOT = next((p for p in (start, *start.parents) if (p/'src'/'config.py').is_file() and (p/'scripts'/'fixed_holdout_manifest.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('src/config.py가 있는 프로젝트 폴더에서 실행하세요.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# 노트북의 현재 작업 폴더와 무관하게 프로젝트 스크립트 로딩
import importlib.util
manifest_path = ROOT / 'scripts' / 'fixed_holdout_manifest.py'
spec = importlib.util.spec_from_file_location('fixed_holdout_manifest', manifest_path)
holdout_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(holdout_module)
assert holdout_module.ROOT.resolve() == ROOT.resolve(), '스크립트의 ROOT 경로를 확인하세요.'
from src.config import DATA_DIR, SEED, set_seed
from src.data_loader import load_data, split_xy, encode_target
from src.preprocessing import binarize_mutations
from src.validation import get_folds, run_cv
from src.evaluation import macro_f1, save_predictions, log_result
from src.features import mutation_count, mutation_type_counts, amino_acid_pair_counts, gene_truncation_flags

set_seed()
print('ROOT:',ROOT)


ROOT: C:\Users\user\Desktop\AI_Projects\ah07-cancer-subtype


In [2]:
# train만 로드: test 파일 접근하지 않음
train_path = DATA_DIR / 'train.csv'
if not train_path.is_file():
    candidates = list(DATA_DIR.rglob('train.csv'))
    if len(candidates) != 1:
        raise FileNotFoundError(f'train.csv 경로 확인 필요: {DATA_DIR} / 후보={candidates}')
    train_path = candidates[0]
train = pd.read_csv(train_path)
X_raw, y_raw = split_xy(train)
y, le = encode_target(y_raw)
assert len(y) == 6201 and len(le.classes_) == 26, (len(y),len(le.classes_))
dev_idx, hold_idx = holdout_module.verify()
assert len(dev_idx)==4960 and len(hold_idx)==1241
assert not np.intersect1d(dev_idx,hold_idx).size
X_dev_raw = X_raw.iloc[dev_idx].reset_index(drop=True)
y_dev = np.asarray(y)[dev_idx]
folds = list(get_folds(y_dev, X=X_dev_raw, classes=le.classes_))
assert len(folds)==5
seen = np.concatenate([va for _,va in folds])
assert len(seen)==len(y_dev) and len(np.unique(seen))==len(y_dev)
print('Dev:',X_dev_raw.shape,'Fold:',[len(va) for _,va in folds])


VERIFIED: dev=4960, holdout=1241; same IDs and row signatures
Dev: (4960, 4384) Fold: [992, 992, 992, 992, 992]


In [3]:
# 시윤님 03_hotspot_bins.ipynb의 E+Position-bin 함수 재현 (핫스팟 미사용)
import re
from collections import Counter
from scipy.sparse import hstack
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

def make_E(X):
    """팀 11번 노트북과 동일한 Feature E 구성 (입력 X만 사용)."""
    parts = [
        binarize_mutations(X),
        np.log1p(mutation_count(X)),
        np.log1p(mutation_type_counts(X)),
        np.log1p(amino_acid_pair_counts(X)),
        gene_truncation_flags(X),
    ]
    out = pd.concat(parts, axis=1).astype(np.float32)
    assert not out.columns.has_duplicates, 'Feature E 중복 열'
    assert out.index.equals(X.index), 'Feature E 인덱스 불일치'
    assert np.isfinite(out.to_numpy()).all(), 'Feature E 비유한값 발생'
    return out

def make_features_E_sparse(X):
    return csr_matrix(make_E(X).to_numpy(dtype=np.float32))

def extract_position(variant):
    match = re.search(r"\d+", variant)
    return int(match.group()) if match else None
# Position-bin 후보 생성 함수

from collections import defaultdict

def extract_position_bins(X, bin_size=50):
    rows = []
    cols = []
    gene_bins = set()

    for row_idx, (_, row) in enumerate(X.iterrows()):
        patient_bins = set()

        for gene, value in row.items():
            if gene == "ID":
                continue

            if pd.isna(value) or str(value) == "WT":
                continue

            for variant in str(value).split():
                pos = extract_position(variant)

                if pos is not None:
                    feature = f"{gene}_bin_{pos // bin_size}"
                    patient_bins.add(feature)

        for feature in patient_bins:
            rows.append(row_idx)
            cols.append(feature)
            gene_bins.add(feature)

    return rows, cols, gene_bins


# Fold별 Position-bin 변환기 구현

from sklearn.base import BaseEstimator, TransformerMixin
from scipy.sparse import csr_matrix
from collections import Counter

class PositionBinTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, bin_size=50, min_patients=5):
        self.bin_size = bin_size
        self.min_patients = min_patients

    def fit(self, X, y=None):
        _, cols, _ = extract_position_bins(
            X, bin_size=self.bin_size
        )

        counts = Counter(cols)

        self.selected_bins_ = sorted(
            feature
            for feature, count in counts.items()
            if count >= self.min_patients
        )

        self.bin_to_idx_ = {
            feature: idx
            for idx, feature in enumerate(self.selected_bins_)
        }

        return self

    def transform(self, X):
        rows, cols, _ = extract_position_bins(
            X, bin_size=self.bin_size
        )

        selected_rows = []
        selected_cols = []

        for row_idx, feature in zip(rows, cols):
            col_idx = self.bin_to_idx_.get(feature)

            if col_idx is not None:
                selected_rows.append(row_idx)
                selected_cols.append(col_idx)

        data = np.ones(len(selected_rows), dtype=np.float32)

        return csr_matrix(
            (data, (selected_rows, selected_cols)),
            shape=(len(X), len(self.selected_bins_)),
            dtype=np.float32
        )# 피처 구성 E + Position-bin CV

from scipy.sparse import hstack

class FeaturesEWithPositionBins(BaseEstimator, TransformerMixin):
    def __init__(self, bin_size=50, min_patients=5):
        self.bin_size = bin_size
        self.min_patients = min_patients

    def fit(self, X, y=None):
        self.position_transformer_ = PositionBinTransformer(
            bin_size=self.bin_size,
            min_patients=self.min_patients
        )
        self.position_transformer_.fit(X)
        return self

    def transform(self, X):
        # 기존 피처 구성 E
        X_base = make_features_E_sparse(X)

        # Fold 학습 데이터에서 선택한 Position-bin
        X_position = self.position_transformer_.transform(X)

        return hstack(
            [X_base, X_position],
            format="csr",
            dtype=np.float32
        )


pipeline_E_position_LR = Pipeline([
    ("features", FeaturesEWithPositionBins(
        bin_size=50,
        min_patients=5
    )),
    ("model", LogisticRegression(
        C=0.1,
        class_weight="balanced",
        max_iter=3000,
        solver="lbfgs"
    ))
])



In [4]:
# 유전자별 Hotspot 아미노산 위치
HOTSPOTS = {
    "IDH1": ["R132"],
    "IDH2": ["R140", "R172"],
    "BRAF": ["V600"],
    "HRAS": ["Q61", "G13", "G12"],
    "KIT": ["D816", "N822", "K642"],
    "CTNNB1": ["S33", "S37", "S45", "T41", "G34", "D32"],
    "U2AF1": ["S34"],
    "PIK3CA": ["E545", "H1047", "E542"],
    "EGFR": ["L858", "G598", "A289"],
    "RET": ["M918"],
    "RAC1": ["P29"],
    "SPOP": ["F133", "W131", "F102", "Y87"],
    "EZH2": ["Y641", "Y646"],
    "TP53": ["R175", "R248", "R273"],
    "PTEN": ["R130"],
}


assert sum(map(len,HOTSPOTS.values())) == 35

In [5]:
def make_hotspot_features(X):
    hotspot_data = {}

    # 일반 Hotspot
    for gene, positions in HOTSPOTS.items():
        if gene not in X.columns:
            raise ValueError(f"유전자 컬럼 없음: {gene}")

        tokens = X[gene].fillna("WT").astype(str).str.split()

        for position in positions:
            col_name = f"hotspot_{gene}_{position}"

            hotspot_data[col_name] = tokens.map(
                lambda variants: int(
                    any(v.startswith(position) for v in variants)
                )
            ).astype(np.int8)

    # NPM1 frameshift
    if "NPM1" not in X.columns:
        raise ValueError("유전자 컬럼 없음: NPM1")

    hotspot_data["hotspot_NPM1_fs"] = (
        X["NPM1"]
        .fillna("WT")
        .astype(str)
        .str.contains("fs", case=False, regex=False)
        .astype(np.int8)
    )

    return pd.DataFrame(hotspot_data, index=X.index)




In [6]:
from lightgbm import LGBMClassifier
from sklearn.base import BaseEstimator, TransformerMixin
from collections import Counter

# 변이 토큰의 시작에 있는 단일 영문자 + 숫자만 인식: V600E -> V600, R132H -> R132.
# 468_469LG처럼 숫자로 시작하거나 복잡한 표기는 의도적으로 제외.
EXACT_POSITION_RE = re.compile(r'^([A-Za-z])(\d+)')

def patient_exact_positions(X):
    result = []
    for row in X.itertuples(index=False, name=None):
        found = set()
        for gene, value in zip(X.columns, row):
            if gene in ('ID', 'SUBCLASS') or pd.isna(value) or str(value).strip().upper() == 'WT':
                continue
            for token in str(value).split():
                match = EXACT_POSITION_RE.match(token)
                if match:
                    found.add(f'{gene}_{match.group(1).upper()}{match.group(2)}')
        result.append(found)
    return result

class ExactPositionTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, min_patients=5):
        self.min_patients = min_patients

    def fit(self, X, y=None):
        counts = Counter(key for keys in patient_exact_positions(X) for key in keys)
        self.selected_ = sorted(k for k,v in counts.items() if v >= self.min_patients)
        self.mapping_ = {key: i for i,key in enumerate(self.selected_)}
        return self

    def transform(self, X):
        if not hasattr(self, 'mapping_'):
            raise RuntimeError('ExactPositionTransformer.fit()을 먼저 실행하세요')
        rows, cols = [], []
        for i, keys in enumerate(patient_exact_positions(X)):
            for key in keys:
                j = self.mapping_.get(key)
                if j is not None:
                    rows.append(i); cols.append(j)
        return csr_matrix((np.ones(len(rows),dtype=np.float32),(rows,cols)),
                          shape=(len(X),len(self.selected_)),dtype=np.float32)

class FeaturesEWithExactPositions(BaseEstimator, TransformerMixin):
    def __init__(self, min_patients=5):
        self.min_patients = min_patients
    def fit(self, X, y=None):
        self.exact_ = ExactPositionTransformer(self.min_patients).fit(X)
        return self
    def transform(self, X):
        return hstack([make_features_E_sparse(X), self.exact_.transform(X)],
                      format='csr',dtype=np.float32)



In [7]:
class FeaturesEHotspot3BinExact(BaseEstimator, TransformerMixin):
    def __init__(self, bin_size=50, min_patients=5, hotspot_multiplier=3):
        self.bin_size=bin_size
        self.min_patients=min_patients
        self.hotspot_multiplier=hotspot_multiplier

    def fit(self, X, y=None):
        # 선택은 반드시 이 fit에 전달된 학습 Fold 데이터에서만 수행
        self.position_transformer_ = PositionBinTransformer(
            bin_size=self.bin_size, min_patients=self.min_patients).fit(X)
        self.exact_transformer_ = ExactPositionTransformer(
            min_patients=self.min_patients).fit(X)
        return self

    def transform(self, X):
        base = make_features_E_sparse(X)
        hotspot = csr_matrix(make_hotspot_features(X).to_numpy(dtype=np.float32)
                             * self.hotspot_multiplier)
        bins = self.position_transformer_.transform(X)
        exact = self.exact_transformer_.transform(X)
        return hstack([base, hotspot, bins, exact], format='csr', dtype=np.float32)

G3_PARAMS = dict(n_estimators=800, learning_rate=0.02, num_leaves=15,
                 min_child_samples=5, colsample_bytree=0.2, subsample=0.8,
                 subsample_freq=1, class_weight='balanced', random_state=42,
                 verbosity=-1, n_jobs=-1)
lgbm_g3 = Pipeline([
    ('features', FeaturesEHotspot3BinExact(bin_size=50, min_patients=5, hotspot_multiplier=3)),
    ('model', LGBMClassifier(**G3_PARAMS))
])
_demo = pd.DataFrame({'BRAF':['V600E','V600K','WT','V600E','V600E','V600E'], 'IDH1':['WT']*6})
assert ExactPositionTransformer(5).fit(_demo).selected_ == ['BRAF_V600']
print('g3 피처 변환기 준비 완료')

g3 피처 변환기 준비 완료


In [8]:
from scipy.sparse import hstack
from sklearn.pipeline import Pipeline

class FeaturesEHotspot3Bins(BaseEstimator, TransformerMixin):
    def __init__(self, bin_size=50, min_patients=5, hotspot_multiplier=3):
        self.bin_size=bin_size
        self.min_patients=min_patients
        self.hotspot_multiplier=hotspot_multiplier
    def fit(self, X, y=None):
        self.bins_=PositionBinTransformer(self.bin_size,self.min_patients).fit(X)
        return self
    def transform(self, X):
        e=make_features_E_sparse(X)
        hotspot=csr_matrix(make_hotspot_features(X).to_numpy(dtype=np.float32))
        bins=self.bins_.transform(X)
        return hstack([e,hotspot*self.hotspot_multiplier,bins],format='csr',dtype=np.float32)

lr_hotspot=Pipeline([
    ('features',FeaturesEHotspot3Bins(50,5,3)),
    ('model',LogisticRegression(C=0.1,class_weight='balanced',max_iter=3000,solver='lbfgs'))
])
K=len(le.classes_)
XGB_PARAMS=dict(
    n_estimators=250,max_depth=3,learning_rate=0.05,
    subsample=0.8,colsample_bytree=0.8,
    objective='multi:softprob',num_class=K,eval_metric='mlogloss',
    tree_method='hist',device='cpu',random_state=SEED,n_jobs=-1
)
xgb_e=XGBClassifier(**XGB_PARAMS)
xgb_position=Pipeline([
    ('features',FeaturesEHotspot3BinExact(bin_size=50,min_patients=5,hotspot_multiplier=3)),
    ('model',XGBClassifier(**XGB_PARAMS))
])
lgbm_g3=Pipeline([
    ('features',FeaturesEHotspot3BinExact(bin_size=50,min_patients=5,hotspot_multiplier=3)),
    ('model',LGBMClassifier(**G3_PARAMS))
])
lr_position=pipeline_E_position_LR

def invsqrt_weights(yy):
    yy=np.asarray(yy,dtype=int)
    counts=np.bincount(yy,minlength=K)
    if np.any(counts==0): raise ValueError('학습 fold에 누락된 클래스')
    weights=1/np.sqrt(counts[yy])
    return (weights/weights.mean()).astype(np.float32)

def checked_proba(p,n,where):
    p=np.asarray(p,dtype=np.float64)
    assert p.shape==(n,K),f'{where} shape={p.shape} expected={(n,K)}'
    assert np.isfinite(p).all() and np.allclose(p.sum(axis=1),1,atol=1e-4),where
    return p

def fit_model(model,X_train,y_train,weighted=False,pipeline=False):
    params=({'model__sample_weight':invsqrt_weights(y_train)} if pipeline else
            {'sample_weight':invsqrt_weights(y_train)}) if weighted else {}
    fitted=clone(model).fit(X_train,y_train,**params)
    assert np.array_equal(fitted.classes_,np.arange(K)), '클래스 순서 불일치'
    return fitted

def cv_predict(model,X,labels,cv_folds,weighted=False,pipeline=False):
    oof=np.zeros((len(labels),K),dtype=np.float64)
    seen=np.zeros(len(labels),dtype=np.int8)
    for i,(tr,va) in enumerate(cv_folds,1):
        m=fit_model(model,X.iloc[tr],np.asarray(labels)[tr],weighted,pipeline)
        oof[va]=checked_proba(m.predict_proba(X.iloc[va]),len(va),f'fold {i}')
        seen[va]+=1
        print(f'  fold {i}/{len(cv_folds)} 완료')
    assert np.all(seen==1), 'OOF coverage 실패'
    return checked_proba(oof,len(labels),'OOF')

def scores_by_fold(proba,labels,cv_folds):
    return np.array([
        f1_score(np.asarray(labels)[va],proba[va].argmax(1),average='macro',zero_division=0)
        for _,va in cv_folds
    ])
print('두 후보 모델 준비 완료 — XGB CPU')


두 후보 모델 준비 완료 — XGB CPU


## A. XGBoost 클래스 가중치 비교

`α=0.5`를 현재 기준으로 두고 같은 모델/동일 Fold에서 α만 바꿉니다. **OOF 예측은 각 fold에서 한 번씩만 생성**합니다. 실험 탐색 결과로 동일 OOF를 반복 선택하면 선택 편향이 생길 수 있으므로 Holdout 최종 검증 전에는 채택 확정하지 않습니다.

In [9]:
from sklearn.metrics import classification_report
from sklearn.metrics import f1_score
from pathlib import Path
import time

OUT = ROOT/'experiments'/'juhyung_xgb_weight_position'
OUT.mkdir(parents=True,exist_ok=True)
K=len(le.classes_)

def weights_alpha(y_train,alpha):
    cnt=np.bincount(np.asarray(y_train,dtype=int),minlength=K)
    if np.any(cnt==0): raise ValueError('학습 fold에 없는 클래스가 있습니다')
    w=cnt[np.asarray(y_train,dtype=int)].astype(np.float64)**(-alpha)
    return (w/w.mean()).astype(np.float32)

def fit_predict_fold(model, X_tr, yy, X_va, alpha):
    fit_kw={'model__sample_weight':weights_alpha(yy,alpha)}
    fitted=clone(model).fit(X_tr,yy,**fit_kw)
    p=fitted.predict_proba(X_va)
    return checked_proba(p,len(X_va),'fold')

def evaluate_model(name, model, alpha, baseline=None):
    oof=np.zeros((len(y_dev),K),dtype=np.float32)
    fold_rows=[]
    for f,(tr,va) in enumerate(folds,1):
        t=time.time()
        p=fit_predict_fold(model,X_dev_raw.iloc[tr],y_dev[tr],X_dev_raw.iloc[va],alpha)
        oof[va]=p
        pred=p.argmax(axis=1)
        fold_rows.append({'experiment':name,'fold':f,'macro_f1':f1_score(y_dev[va],pred,average='macro',zero_division=0),'seconds':round(time.time()-t,1)})
        print(f'{name} fold {f}/5 F1={fold_rows[-1]["macro_f1"]:.5f} ({fold_rows[-1]["seconds"]}s)')
    overall=f1_score(y_dev,oof.argmax(1),average='macro',zero_division=0)
    mean=np.mean([r['macro_f1'] for r in fold_rows])
    print(f'{name}: fold mean={mean:.5f}, pooled OOF={overall:.5f}')
    return oof,fold_rows,{'experiment':name,'alpha':alpha,'fold_mean_macro_f1':mean,'pooled_oof_macro_f1':overall}

RUN_WEIGHTS=True
RUN_POSITION=True  # 먼저 가중치만 실행하려면 False
alphas=[0.5,0.75,1.0]
weight_results={}; rows=[]; fold_rows=[]
if RUN_WEIGHTS:
    for alpha in alphas:
        name=f'weight_a{alpha}'
        p,fr,r=evaluate_model(name,xgb_position,alpha)
        weight_results[name]=p; rows.append(r); fold_rows.extend(fr)
    baseline_p=weight_results['weight_a0.5']
    baseline_fold_mean=next(r['fold_mean_macro_f1'] for r in rows if r['experiment']=='weight_a0.5')
    for r in rows:r['delta_vs_baseline_fold_mean']=r['fold_mean_macro_f1']-baseline_fold_mean
    display(pd.DataFrame(rows))


weight_a0.5 fold 1/5 F1=0.50198 (99.9s)
weight_a0.5 fold 2/5 F1=0.46749 (99.5s)
weight_a0.5 fold 3/5 F1=0.48201 (96.0s)
weight_a0.5 fold 4/5 F1=0.49034 (103.6s)
weight_a0.5 fold 5/5 F1=0.49050 (97.8s)
weight_a0.5: fold mean=0.48646, pooled OOF=0.48977
weight_a0.75 fold 1/5 F1=0.49366 (98.6s)
weight_a0.75 fold 2/5 F1=0.46384 (94.7s)
weight_a0.75 fold 3/5 F1=0.47594 (95.7s)
weight_a0.75 fold 4/5 F1=0.49373 (93.7s)
weight_a0.75 fold 5/5 F1=0.48589 (91.6s)
weight_a0.75: fold mean=0.48261, pooled OOF=0.48642
weight_a1.0 fold 1/5 F1=0.49516 (94.0s)
weight_a1.0 fold 2/5 F1=0.45785 (93.3s)
weight_a1.0 fold 3/5 F1=0.47867 (96.4s)
weight_a1.0 fold 4/5 F1=0.47633 (94.3s)
weight_a1.0 fold 5/5 F1=0.47821 (92.4s)
weight_a1.0: fold mean=0.47724, pooled OOF=0.48010


,experiment,alpha,fold_mean_macro_f1,pooled_oof_macro_f1,delta_vs_baseline_fold_mean
0,weight_a0.5,0.50,0.486463,0.489767,0.000000
1,weight_a0.75,0.75,0.482615,0.486419,-0.003848
2,weight_a1.0,1.00,0.477242,0.480101,-0.009220


## B. 절단 변이 상대 위치 구간 피처

변이 문자열에서 숫자 위치를 추출합니다. `*`, `Ter`, `fs`, `frameshift`, `stop_gained`, `nonsense` 표기를 **절단 변이 후보**로 간주합니다. 원시 문자열의 표기 방식에 따라 탐지율이 달라질 수 있으므로 아래 탐지율을 반드시 점검하세요. 외부 유전자 길이는 사용하지 않습니다.

각 유전자의 최대 관측 위치는 **학습 fold에서 관측한 모든 위치가 있는 변이**를 기준으로 계산합니다. 검증 위치가 학습 최대를 넘으면 마지막 bin으로 클리핑합니다. 최대 위치가 없는 유전자는 별도 미지수 카운트로 기록합니다.


In [10]:
import re
from sklearn.base import BaseEstimator, TransformerMixin
from scipy.sparse import csr_matrix,hstack
POS_RE=re.compile(r'(?<!\d)(\d{1,7})(?!\d)')
TRUNC_RE=re.compile(r'(?:\*|\bter\b|fs|frameshift|stop[_ -]?gained|nonsense)',re.IGNORECASE)

def extract_events(X):
    """행별 (gene, position, truncation) 이벤트. split delimiter는 공백/쉼표/세미콜론."""
    gene_names=list(X.columns)
    rows=[]
    for vals in X.itertuples(index=False,name=None):
        ev=[]
        for gene,val in zip(gene_names,vals):
            if pd.isna(val):continue
            s=str(val).strip()
            if not s or s.upper() in ('WT','NAN','NONE'):continue
            for token in re.split(r'[\s,;]+',s):
                if not token:continue
                m=POS_RE.search(token)
                if m:
                    ev.append((gene,int(m.group(1)),bool(TRUNC_RE.search(token))))
        rows.append(ev)
    return rows

class RelativeTruncationBins(BaseEstimator,TransformerMixin):
    def __init__(self,n_bins=6):self.n_bins=n_bins
    def fit(self,X,y=None):
        events=extract_events(X)
        self.maxpos_={}
        for row in events:
            for gene,pos,_ in row:
                self.maxpos_[gene]=max(pos,self.maxpos_.get(gene,0))
        self.train_events_=sum(len(row) for row in events)
        self.train_trunc_events_=sum(t for row in events for _,_,t in row)
        return self
    def transform(self,X):
        if not hasattr(self,'maxpos_'):raise RuntimeError('fit first')
        mat=np.zeros((len(X),self.n_bins+3),dtype=np.float32)
        for i,row in enumerate(extract_events(X)):
            for gene,pos,is_trunc in row:
                if not is_trunc:continue
                mat[i,self.n_bins]+=1  # total truncation events
                denom=self.maxpos_.get(gene,0)
                if denom<=0:
                    mat[i,self.n_bins+1]+=1  # unseen gene
                    continue
                r=min(max(pos/denom,0.0),1.0)
                b=min(int(r*self.n_bins),self.n_bins-1)
                mat[i,b]+=1
            mat[i,self.n_bins+2]=float(mat[i,self.n_bins]>0)
        return csr_matrix(np.log1p(mat),dtype=np.float32)

class EHotspotPositionWithRelative(BaseEstimator,TransformerMixin):
    def __init__(self,n_bins=6,bin_size=50,min_patients=5,hotspot_multiplier=3):
        self.n_bins=n_bins;self.bin_size=bin_size;self.min_patients=min_patients;self.hotspot_multiplier=hotspot_multiplier
    def fit(self,X,y=None):
        self.base_=FeaturesEHotspot3BinExact(self.bin_size,self.min_patients,self.hotspot_multiplier).fit(X,y)
        self.relative_=RelativeTruncationBins(self.n_bins).fit(X,y)
        return self
    def transform(self,X):
        return hstack([self.base_.transform(X),self.relative_.transform(X)],format='csr',dtype=np.float32)

def position_model(n_bins):
    return Pipeline([('features',EHotspotPositionWithRelative(n_bins=n_bins)),('model',XGBClassifier(**XGB_PARAMS))])

# 데이터 포맷 점검: train/dev의 변이 표기에서 절단 후보가 실제로 추출되는가?
sample=X_dev_raw.iloc[:min(200,len(X_dev_raw))]
events=extract_events(sample)
print('샘플 변이 이벤트:',sum(map(len,events)),'절단 후보:',sum(t for row in events for _,_,t in row))
print('샘플 절단 토큰이 0개라면 원시 표기를 확인하고 TRUNC_RE를 수정해야 합니다.')


샘플 변이 이벤트: 7525 절단 후보: 769
샘플 절단 토큰이 0개라면 원시 표기를 확인하고 TRUNC_RE를 수정해야 합니다.


In [11]:
position_results={}
if RUN_POSITION:
    if 'baseline_p' not in globals():
        baseline_p,fr,r=evaluate_model('weight_a0.5',xgb_position,0.5)
        weight_results['weight_a0.5']=baseline_p;rows.append(r);fold_rows.extend(fr)
    baseline_fold_mean=np.mean([r['macro_f1'] for r in fold_rows if r['experiment']=='weight_a0.5'])
    for bins in [4,6,8,10]:
        name=f'relpos_{bins}bins_a0.5'
        p,fr,r=evaluate_model(name,position_model(bins),0.5)
        position_results[name]=p; rows.append(r);fold_rows.extend(fr)
    for r in rows:r['delta_vs_baseline_fold_mean']=r['fold_mean_macro_f1']-baseline_fold_mean
    display(pd.DataFrame(rows).sort_values('fold_mean_macro_f1',ascending=False))


relpos_4bins_a0.5 fold 1/5 F1=0.49734 (107.6s)
relpos_4bins_a0.5 fold 2/5 F1=0.46119 (106.7s)
relpos_4bins_a0.5 fold 3/5 F1=0.47891 (107.7s)
relpos_4bins_a0.5 fold 4/5 F1=0.48758 (108.1s)
relpos_4bins_a0.5 fold 5/5 F1=0.48988 (107.6s)
relpos_4bins_a0.5: fold mean=0.48298, pooled OOF=0.48667
relpos_6bins_a0.5 fold 1/5 F1=0.49968 (110.8s)
relpos_6bins_a0.5 fold 2/5 F1=0.45702 (117.3s)
relpos_6bins_a0.5 fold 3/5 F1=0.47771 (119.9s)
relpos_6bins_a0.5 fold 4/5 F1=0.49099 (109.1s)
relpos_6bins_a0.5 fold 5/5 F1=0.49344 (108.9s)
relpos_6bins_a0.5: fold mean=0.48377, pooled OOF=0.48745
relpos_8bins_a0.5 fold 1/5 F1=0.50247 (110.5s)
relpos_8bins_a0.5 fold 2/5 F1=0.46356 (106.4s)
relpos_8bins_a0.5 fold 3/5 F1=0.47796 (107.1s)
relpos_8bins_a0.5 fold 4/5 F1=0.48858 (113.4s)
relpos_8bins_a0.5 fold 5/5 F1=0.49353 (110.5s)
relpos_8bins_a0.5: fold mean=0.48522, pooled OOF=0.48958
relpos_10bins_a0.5 fold 1/5 F1=0.50428 (116.5s)
relpos_10bins_a0.5 fold 2/5 F1=0.45955 (131.6s)
relpos_10bins_a0.5 fold 3/5 

,experiment,alpha,fold_mean_macro_f1,pooled_oof_macro_f1,delta_vs_baseline_fold_mean
0,weight_a0.5,0.50,0.486463,0.489767,0.000000
5,relpos_8bins_a0.5,0.50,0.485221,0.489584,-0.001242
6,relpos_10bins_a0.5,0.50,0.484792,0.488380,-0.001670
4,relpos_6bins_a0.5,0.50,0.483766,0.487452,-0.002697
3,relpos_4bins_a0.5,0.50,0.482981,0.486673,-0.003482
1,weight_a0.75,0.75,0.482615,0.486419,-0.003848
2,weight_a1.0,1.00,0.477242,0.480101,-0.009220


## C. 클래스별 F1 / 함께 틀린 환자 / 결과 저장

`baseline_p`와 비교하며, 현재 실행의 후보 XGB가 맞힌 환자 중 **기준 XGB가 틀린 환자** 비율을 계산합니다. 팀에서 요청한 **기존 세 모델 모두 오답(쌍둥이 제외)** 지표는 동일 Dev OOF의 LR/g3 파일과 충돌 마스크가 있어야 계산 가능하므로 여기서는 임의로 만들지 않습니다. 또한 같은 Dev OOF를 여러 번 탐색한 결과는 최종 독립 검증이 아닙니다.

In [12]:
from sklearn.metrics import precision_recall_fscore_support

def class_f1(p):
    return precision_recall_fscore_support(y_dev,p.argmax(1),labels=np.arange(K),zero_division=0)[2]

all_preds={**weight_results,**position_results}
base=all_preds['weight_a0.5']
base_correct=base.argmax(1)==y_dev
class_rows=[]; recovery_rows=[]
for name,p in all_preds.items():
    f=class_f1(p);f0=class_f1(base)
    for j,cl in enumerate(le.classes_):
        class_rows.append({'experiment':name,'class':cl,'f1':f[j],'baseline_f1':f0[j],'delta_f1':f[j]-f0[j]})
    correct=p.argmax(1)==y_dev
    recovery_rows.append({'experiment':name,'baseline_xgb_wrong_n':int((~base_correct).sum()),'candidate_recovers_n':int((~base_correct & correct).sum()),'recovery_rate':float((~base_correct & correct).sum()/max((~base_correct).sum(),1))})

summary=pd.DataFrame(rows).drop_duplicates('experiment')
fold_df=pd.DataFrame(fold_rows).drop_duplicates(['experiment','fold'])
class_df=pd.DataFrame(class_rows)
recovery_df=pd.DataFrame(recovery_rows)
summary.to_csv(OUT/'summary.csv',index=False,encoding='utf-8-sig')
fold_df.to_csv(OUT/'fold_scores.csv',index=False,encoding='utf-8-sig')
class_df.to_csv(OUT/'class_f1.csv',index=False,encoding='utf-8-sig')
recovery_df.to_csv(OUT/'xgb_recovery.csv',index=False,encoding='utf-8-sig')
for name,p in all_preds.items():np.save(OUT/f'{name}__dev_oof.npy',p)
print('결과 저장:',OUT)
display(summary.sort_values('fold_mean_macro_f1',ascending=False))
display(class_df[class_df['class'].isin(['KIPAN','KIRC','GBMLGG','LGG'])].pivot(index='class',columns='experiment',values='f1'))
display(recovery_df)
print('단일 XGB 통과 후보 (fold 평균 +0.003 이상):',summary.loc[summary.delta_vs_baseline_fold_mean>=0.003,'experiment'].tolist())


결과 저장: C:\Users\user\Desktop\AI_Projects\ah07-cancer-subtype\experiments\juhyung_xgb_weight_position


,experiment,alpha,fold_mean_macro_f1,pooled_oof_macro_f1,delta_vs_baseline_fold_mean
0,weight_a0.5,0.50,0.486463,0.489767,0.000000
5,relpos_8bins_a0.5,0.50,0.485221,0.489584,-0.001242
6,relpos_10bins_a0.5,0.50,0.484792,0.488380,-0.001670
4,relpos_6bins_a0.5,0.50,0.483766,0.487452,-0.002697
3,relpos_4bins_a0.5,0.50,0.482981,0.486673,-0.003482
1,weight_a0.75,0.75,0.482615,0.486419,-0.003848
2,weight_a1.0,1.00,0.477242,0.480101,-0.009220


experiment,relpos_10bins_a0.5,relpos_4bins_a0.5,relpos_6bins_a0.5,relpos_8bins_a0.5,weight_a0.5,weight_a0.75,weight_a1.0
class,,,,,,,
GBMLGG,0.463158,0.470588,0.457576,0.455235,0.465465,0.436066,0.400729
KIPAN,0.375000,0.372642,0.364277,0.365566,0.363420,0.339869,0.300752
KIRC,0.350598,0.352250,0.355469,0.370370,0.365422,0.372439,0.409506
LGG,0.557143,0.548077,0.557377,0.557377,0.552885,0.568233,0.582278


,experiment,baseline_xgb_wrong_n,candidate_recovers_n,recovery_rate
0,weight_a0.5,2511,0,0.000000
1,weight_a0.75,2511,91,0.036241
2,weight_a1.0,2511,152,0.060534
3,relpos_4bins_a0.5,2511,76,0.030267
4,relpos_6bins_a0.5,2511,83,0.033055
5,relpos_8bins_a0.5,2511,81,0.032258
6,relpos_10bins_a0.5,2511,91,0.036241


단일 XGB 통과 후보 (fold 평균 +0.003 이상): []
